# Data Exploration: Skin Lesion Segmentation

This notebook explores the ISIC 2018 dataset and visualizes sample images with their corresponding segmentation masks.

In [ ]:
# Mount Google Drive (Google Colab only)
import sys

# Check if running on Google Colab or Local machine
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive has been mounted successfully")
    except Exception as e:
        print(f"Failed to mount Google Drive: {e}")
        IS_COLAB = False
else:
    print("Running on local machine (not Google Colab)")

In [ ]:
import os

# Handle src path for both local and Colab environments
if 'google.colab' in sys.modules:
    # Google Colab: src files are copied
    sys.path.insert(0, '/content/drive/MyDrive/CSCI 425/data/models/notebooks/src')
else:
    # Local machine: src is in the notebooks directory
    sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import torch
import matplotlib.pyplot as plt
from dataset import SegmentationDatasets
from utils import show_samples

In [ ]:
# Set up data paths
# Handle both local and Google Colab environments
if IS_COLAB:
    # Google Colab path - adjust the path to match your Drive structure
    # Example: /content/drive/MyDrive/path/
    base_dir = '/content/drive/MyDrive/CSCI 425/data/'
    print("Using Google Colab data path")
else:
    # Local machine path
    base_dir = os.path.join(os.getcwd(), '../../')
    print("Using local machine data path")

train_image_dir = os.path.join(base_dir, 'train/images')
train_mask_dir = os.path.join(base_dir, 'train/masks')
val_image_dir = os.path.join(base_dir, 'val/images')
val_mask_dir = os.path.join(base_dir, 'val/masks')

print(f"Training images: {train_image_dir}")
print(f"Training masks: {train_mask_dir}")
print(f"Validation images: {val_image_dir}")
print(f"Validation masks: {val_mask_dir}")

In [ ]:
# Load training dataset without transforms to view original data
train_dataset = SegmentationDatasets(
    image_dir=train_image_dir,
    mask_dir=train_mask_dir,
    transform=None
)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Sample image shape: {train_dataset[0][0].shape}")
print(f"Sample mask shape: {train_dataset[0][1].shape}")

## Sample Images and Masks Visualization

Below are sample images from the training dataset with their corresponding segmentation masks:

In [ ]:
# Visualize sample images and masks from the training dataset
show_samples(train_dataset, n=6)

## Model Architecture: U-Net with ResNet-34 Encoder

Load a pretrained U-Net model using segmentation-models-pytorch library. The encoder uses ResNet-34 pretrained on ImageNet for transfer learning.

In [ ]:
import segmentation_models_pytorch as smp

# Load pretrained U-Net with ResNet-34 encoder
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',  # Use ImageNet pretrained weights
    in_channels=3,                # RGB images
    classes=1                     # Binary segmentation (lesion vs non-lesion)
)

print(f"Model loaded successfully!")
print(f"Model type: {type(model)}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Loss Function: Combined BCE + Dice Loss

For imbalanced segmentation tasks, we use a combined loss function that leverages both:
- **Binary Cross-Entropy (BCE)**: Handles pixel-level classification errors
- **Dice Loss**: Focuses on overall shape and overlap between predicted and target masks

This combination is particularly effective when the foreground (lesion) is much smaller than the background.

In [ ]:
from loss import BCEWithDiceLoss, DiceLoss

# Initialize combined loss function with equal weighting (α = 0.5)
criterion = BCEWithDiceLoss(alpha=0.5)

print("Loss function initialized successfully!")
print(f"Loss function: {type(criterion).__name__}")
print(f"Alpha (BCE weight): {criterion.alpha}")
print(f"Dice weight: {1 - criterion.alpha}")

# Test the loss with dummy data
batch_size = 4
height, width = 256, 256

# Create dummy predictions and targets
dummy_pred = torch.randn(batch_size, 1, height, width)  # Logits
dummy_target = torch.randint(0, 2, (batch_size, 1, height, width)).float()  # Binary mask

# Compute loss
test_loss = criterion(dummy_pred, dummy_target)
print(f"\nTest loss computation: {test_loss.item():.4f}")
print("Loss function is working correctly!")

## Optimizer and Learning Rate Scheduler

Configure the Adam optimizer with an initial learning rate of 1e-4 and cosine annealing learning rate schedule for smooth learning rate decay during training.

In [ ]:
# Training hyperparameters
initial_lr = 1e-4
num_epochs = 50  # Full training with up to 50 epochs and early stopping
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {device}")
print(f"Total number of Epochs: {num_epochs}")
print(f"Using CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Visualize learning rate schedule
lrs = []
temp_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    torch.optim.Adam(model.parameters(), lr=initial_lr),
    T_max=num_epochs,
    eta_min=1e-6
)

for epoch in range(num_epochs):
    lrs.append(temp_scheduler.get_last_lr()[0])
    temp_scheduler.step()

# Plot learning rate schedule
plt.figure(figsize=(10, 5))
plt.plot(lrs, linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.title('Cosine Annealing Learning Rate Schedule', fontsize=14)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

print(f"Learning rate schedule visualization complete!")
print(f"Initial LR: {lrs[0]:.2e}")
print(f"Final LR: {lrs[-1]:.2e}")

In [ ]:
# Initialize optimizer and learning rate scheduler for training
# (Note: Create fresh optimizer after visualizing the schedule)
optimizer = torch.optim.Adam(model.parameters(), lr=initial_lr)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6
)

print(f"\nOptimizer and scheduler initialized for training!")

In [ ]:
# GPU Memory Optimization (important for Google Colab)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache cleared")
    
    # Set mixed precision for faster training (optional, helps with memory)
    # torch.cuda.amp.autocast(enabled=True)
    
print(f"All components ready for training!")
print(f"Starting GPU training on device: {device}")

In [ ]:
# Copy src files from Google Drive to Colab runtime
import shutil

if IS_COLAB:
    print("Copying src files from Google Drive to Colab runtime...")
    src_drive_path = '/content/drive/MyDrive/CSCI 425/data/models/notebooks/src'
    src_runtime_path = '/content/src'
    
    # Check if src exists on Drive and copy it
    if os.path.exists(src_drive_path):
        if os.path.exists(src_runtime_path):
            # Remove the current /src in the Colab runtime directory
            shutil.rmtree(src_runtime_path)
        # Copy the /src folder from Google Drive into the Colab runtime directory
        shutil.copytree(src_drive_path, src_runtime_path)
        print(f"✓ Src files copied to runtime path: {src_runtime_path}")
    else:
        print(f"⚠ Warning: {src_drive_path} not found on Google Drive")
        print("Make sure to reference README.md on how to upload the src folder to your Google Drive")
else:
    print("Running on local machine - src files will be loaded from local path")

## Full Model Training (50 Epochs with Validation)

Train the full U-Net model on the complete training dataset with validation evaluation.

The model trains for up to 50 epochs with early stopping based on validation Dice score. The best model checkpoint is saved based on validation performance.

In [ ]:
from torch.utils.data import DataLoader
from dataset import train_transform, val_transform
from metrics import dice_score

# Training hyperparameters - optimized for GPU
batch_size = 32 if torch.cuda.is_available() else 16  # Larger batches for GPU
num_workers = 4 if IS_COLAB or torch.cuda.is_available() else 0  # Higher num_workers for Linux/Colab

print(f"Batch size: {batch_size} (GPU optimized)" if torch.cuda.is_available() else f"Batch size: {batch_size} (CPU)")
print(f"Num workers: {num_workers}")

# Load training dataset with transforms
train_dataset_with_transforms = SegmentationDatasets(
    image_dir=train_image_dir,
    mask_dir=train_mask_dir,
    transform=train_transform
)

# Load validation dataset with transforms
val_dataset = SegmentationDatasets(
    image_dir=val_image_dir,
    mask_dir=val_mask_dir,
    transform=val_transform
)

# Create data loaders
train_loader = DataLoader(
    train_dataset_with_transforms,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\nData loaders created successfully!")
print(f"Training set: {len(train_dataset_with_transforms)} images")
print(f"Validation set: {len(val_dataset)} images")
print(f"Training batches per epoch: {len(train_loader)}")
print(f"Validation batches per epoch: {len(val_loader)}")

### Sanity Check: Overfit on 10 Samples

Before full training, we run a sanity check by training on just 10 samples. This confirms:
- Model can learn (loss decreases, metrics improve)
- Training pipeline is working correctly
- No issues with data loading or loss computation

The model should quickly overfit to these 10 samples, achieving very high metrics.

In [ ]:
# Sanity check: Overfit on 10 samples
from torch.utils.data import Subset

# Create a small dataset with only 10 samples for sanity check
sanity_dataset = Subset(train_dataset_with_transforms, indices=list(range(10)))
sanity_loader = DataLoader(
    sanity_dataset,
    batch_size=2,
    shuffle=True,
    pin_memory=True if torch.cuda.is_available() else False
)

print("=" * 70)
print("SANITY CHECK: Overfitting on 10 samples")
print("=" * 70)
print(f"Sanity check dataset size: {len(sanity_dataset)}")
print(f"Batches per epoch: {len(sanity_loader)}")

# Reset model and optimizer for sanity check
model_sanity = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1
)
model_sanity = model_sanity.to(device)

optimizer_sanity = torch.optim.Adam(model_sanity.parameters(), lr=initial_lr)

# Train on 10 samples for 50 epochs (should achieve very high metrics)
sanity_history = {
    'loss': [],
    'dice': [],
}

print(f"\n{'Epoch':<6} {'Loss':<12} {'Dice':<12}")
print("-" * 30)

num_sanity_epochs = 50
for epoch in range(num_sanity_epochs):
    model_sanity.train()
    epoch_loss = 0.0
    epoch_dice = 0.0
    batches = 0
    
    for images, masks in sanity_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        if masks.dim() == 3:
            masks = masks.unsqueeze(1)
        
        # Forward pass
        outputs = model_sanity(images)
        loss = criterion(outputs, masks)
        
        # Backward pass
        optimizer_sanity.zero_grad()
        loss.backward()
        optimizer_sanity.step()
        
        # Track metrics
        epoch_loss += loss.item()
        epoch_dice += dice_score(outputs, masks)
        batches += 1
    
    avg_loss = epoch_loss / batches
    avg_dice = epoch_dice / batches
    
    sanity_history['loss'].append(avg_loss)
    sanity_history['dice'].append(avg_dice)
    
    # Print every 10 epochs
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"{epoch+1:<6} {avg_loss:<12.6f} {avg_dice:<12.6f}")

print("\n" + "=" * 70)
print(f"✓ Sanity check complete!")
print(f"Final Loss: {sanity_history['loss'][-1]:.6f}")
print(f"Final Dice: {sanity_history['dice'][-1]:.6f}")
print(f"✓ Model successfully overfits to 10 samples")
print("✓ Training pipeline is working correctly!")
print("=" * 70)

# Plot sanity check results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
ax1.plot(sanity_history['loss'], linewidth=2, color='red')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Sanity Check: Loss Decreases', fontsize=14)
ax1.grid(True, alpha=0.3)

# Dice plot
ax2.plot(sanity_history['dice'], linewidth=2, color='green')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Dice Score', fontsize=12)
ax2.set_title('Sanity Check: Dice Increases', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Ready to proceed with full training!")

In [ ]:
# Full Training loop with validation

# Move model to device before training
model = model.to(device)

history = {
    'train_loss': [],
    'train_dice': [],
    'val_loss': [],
    'val_dice': []
}

best_val_dice = 0.0
patience_counter = 0
patience = 10  # Early stopping patience

print("Starting training...\n")
print(f"{'Epoch':<6} {'Train Loss':<12} {'Train Dice':<12} {'Val Loss':<12} {'Val Dice':<12} {'LR':<12}")
print("-" * 70)

for epoch in range(num_epochs):
    # ===== TRAINING PHASE =====
    model.train()
    train_loss_total = 0.0
    train_dice_total = 0.0
    train_batches = 0
    
    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        # Ensure masks have the correct shape [batch, 1, height, width]
        if masks.dim() == 3:
            masks = masks.unsqueeze(1)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        train_loss_total += loss.item()
        train_dice_total += dice_score(outputs, masks)
        train_batches += 1
    
    # Average training metrics
    avg_train_loss = train_loss_total / train_batches
    avg_train_dice = train_dice_total / train_batches
    
    # ===== VALIDATION PHASE =====
    model.eval()
    val_loss_total = 0.0
    val_dice_total = 0.0
    val_batches = 0
    
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            # Ensure masks have the correct shape [batch, 1, height, width]
            if masks.dim() == 3:
                masks = masks.unsqueeze(1)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            # Track metrics
            val_loss_total += loss.item()
            val_dice_total += dice_score(outputs, masks)
            val_batches += 1
    
    # Average validation metrics
    avg_val_loss = val_loss_total / val_batches
    avg_val_dice = val_dice_total / val_batches
    
    # Store history
    history['train_loss'].append(avg_train_loss)
    history['train_dice'].append(avg_train_dice)
    history['val_loss'].append(avg_val_loss)
    history['val_dice'].append(avg_val_dice)
    
    # Update learning rate
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Print metrics
    print(f"{epoch+1:<6} {avg_train_loss:<12.6f} {avg_train_dice:<12.6f} {avg_val_loss:<12.6f} {avg_val_dice:<12.6f} {current_lr:<12.2e}")
    
    # Early stopping logic
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
    
    # Early stopping check
    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

print("\nTraining completed!")
print(f"Best validation Dice score: {best_val_dice:.6f}")

## Training Results and Analysis

Visualize training curves and summary statistics from the full training run.

In [ ]:
# Plot training curves and display summary statistics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Training and Validation Loss
axes[0, 0].plot(history['train_loss'], linewidth=2, label='Train Loss', color='blue')
axes[0, 0].plot(history['val_loss'], linewidth=2, label='Val Loss', color='red')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Training and Validation Dice Score
axes[0, 1].plot(history['train_dice'], linewidth=2, label='Train Dice', color='green')
axes[0, 1].plot(history['val_dice'], linewidth=2, label='Val Dice', color='orange')
axes[0, 1].axhline(y=0.83, color='red', linestyle='--', linewidth=2, label='Target (0.83)')
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Dice Score', fontsize=12)
axes[0, 1].set_title('Training and Validation Dice Score', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Learning Rate Schedule
num_epochs_trained = len(history['train_loss'])
learning_rates = []
temp_scheduler_viz = torch.optim.lr_scheduler.CosineAnnealingLR(
    torch.optim.Adam([torch.nn.Parameter(torch.randn(1))], lr=initial_lr),
    T_max=50,
    eta_min=1e-6
)
for _ in range(50):
    learning_rates.append(temp_scheduler_viz.get_last_lr()[0])
    temp_scheduler_viz.step()

axes[1, 0].plot(learning_rates[:num_epochs_trained], linewidth=2, color='purple')
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Learning Rate', fontsize=12)
axes[1, 0].set_title('Learning Rate Schedule (Cosine Annealing)', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_yscale('log')

# Plot 4: Summary Statistics (text)
axes[1, 1].axis('off')
summary_text = f"""
TRAINING SUMMARY
{'='*40}

Total Epochs Trained: {len(history['train_loss'])}
Final Epoch: {len(history['train_loss'])}

Best Validation Dice: {best_val_dice:.6f}
Best Epoch: {history['val_dice'].index(max(history['val_dice'])) + 1}

Final Metrics:
  • Train Loss: {history['train_loss'][-1]:.6f}
  • Val Loss: {history['val_loss'][-1]:.6f}
  • Train Dice: {history['train_dice'][-1]:.6f}
  • Val Dice: {history['val_dice'][-1]:.6f}

Target Performance: Dice ≥ 0.83
Status: {'✓ ACHIEVED' if best_val_dice >= 0.83 else '✗ NOT MET'}

Early Stopping:
  • Triggered: {patience_counter >= patience}
  • Patience Counter: {patience_counter}/{patience}
"""

axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
                verticalalignment='center', bbox=dict(boxstyle='round', 
                facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*70)
print("TRAINING RESULTS SUMMARY")
print("="*70)
print(f"✓ Training completed successfully!")
print(f"✓ Best validation Dice score: {best_val_dice:.6f}")
print(f"✓ Model checkpoint saved: best_model.pth")
print(f"✓ Training curves saved: training_results.png")
print("="*70)

In [ ]:
# Save trained model and training history to Google Drive (Colab only)
import json
from datetime import datetime

if IS_COLAB:
    # Create checkpoint directory on Google Drive
    checkpoint_dir = '/content/drive/MyDrive/CSCI 425/checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # Save best model
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    model_save_path = os.path.join(checkpoint_dir, f'best_model_{timestamp}.pth')
    torch.save(model.state_dict(), model_save_path)
    print(f"✓ Model saved to: {model_save_path}")
    
    # Save training history
    history_save_path = os.path.join(checkpoint_dir, f'training_history_{timestamp}.json')
    with open(history_save_path, 'w') as f:
        json.dump(history, f, indent=4)
    print(f"✓ Training history saved to: {history_save_path}")
    
    # Save training summary
    summary = {
        'best_val_dice': float(best_val_dice),
        'total_epochs': len(history['train_loss']),
        'final_train_loss': float(history['train_loss'][-1]),
        'final_val_loss': float(history['val_loss'][-1]),
        'timestamp': timestamp
    }
    summary_path = os.path.join(checkpoint_dir, f'summary_{timestamp}.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=4)
    print(f"✓ Training summary saved to: {summary_path}")
    print(f"\nAll files saved to Google Drive!")
else:
    print("Local machine - saving to current directory")
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    torch.save(model.state_dict(), f'best_model_{timestamp}.pth')
    with open('training_history.json', 'w') as f:
        json.dump(history, f, indent=4)
    print("✓ Model and history saved locally")